In [1]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
from scipy.signal import fftconvolve



Bad key "text.kerning_factor" on line 4 in
C:\Users\mckang\AppData\Local\anaconda3\envs\former_env\lib\site-packages\matplotlib\mpl-data\stylelib\_classic_test_patch.mplstyle.
You probably need to get an updated matplotlibrc file from
http://github.com/matplotlib/matplotlib/blob/master/matplotlibrc.template
or from the matplotlib source distribution


In [2]:
# ================================================================
# User settings
# ================================================================

psf_path = 'psf_used.tif'
image_path = 'Si_highSNR.png'

iteration = 50
eps = 1e-12
noise_level = 10
output_crop_fraction = 0.05

# Central ROI fraction used for RME calculation.
# 0.50 means central 50% of image height and central 50% of image width.
rme_center_fraction = 0.50

output_dir = 'example_deconvolution_results'
iteration_tif_dir = os.path.join(output_dir, 'rl_iteration_tif')

os.makedirs(output_dir, exist_ok=True)
os.makedirs(iteration_tif_dir, exist_ok=True)


In [3]:
# ================================================================
# Helper functions
# ================================================================

def load_psf(path):
    ext = os.path.splitext(path)[1].lower()

    if ext in ('.tif', '.tiff'):
        psf = cv2.imread(path, cv2.IMREAD_UNCHANGED)
        if psf is None:
            raise FileNotFoundError(path)
        if psf.ndim == 3:
            psf = cv2.cvtColor(psf, cv2.COLOR_BGR2GRAY)
    elif ext == '.npy':
        psf = np.load(path)
    elif ext == '.txt':
        psf = np.loadtxt(path)
    else:
        raise ValueError('Supported PSF formats: .tif, .tiff, .npy, .txt')

    psf = np.asarray(psf, dtype=np.float64)

    if psf.ndim != 2:
        raise ValueError('PSF must be 2D: {}'.format(psf.shape))
    if psf.shape[0] != psf.shape[1]:
        raise ValueError('PSF must be square: {}'.format(psf.shape))
    if psf.shape[0] % 2 == 0:
        raise ValueError('PSF size must be odd: {}'.format(psf.shape))
    if not np.all(np.isfinite(psf)):
        raise ValueError('PSF contains NaN or Inf.')

    psf = np.maximum(psf, 0.0)
    s = psf.sum()
    if s <= 0:
        raise ValueError('PSF sum is non-positive.')

    return psf / s


def load_image(path):
    image = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if image is None:
        raise FileNotFoundError(path)
    if image.ndim == 3:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    image = np.asarray(image, dtype=np.float64)
    if not np.all(np.isfinite(image)):
        raise ValueError('Image contains NaN or Inf.')
    if image.max() <= 0:
        raise ValueError('Image maximum is non-positive.')

    return image / image.max()


def correlate_same_fft(image, kernel):
    # Reproduce the direction of tf.nn.conv2d cross-correlation using fftconvolve.
    return fftconvolve(image, kernel[::-1, ::-1], mode='same')


def center_roi(image, fraction=0.50):
    if not 0.0 < fraction <= 1.0:
        raise ValueError('fraction must satisfy 0 < fraction <= 1')

    h, w = image.shape
    roi_h = max(1, int(round(h * fraction)))
    roi_w = max(1, int(round(w * fraction)))

    y0 = (h - roi_h) // 2
    x0 = (w - roi_w) // 2

    return image[y0:y0 + roi_h, x0:x0 + roi_w]


def relative_mean_error(current, initial, center_fraction=0.50, eps=1e-12):
    current_roi = center_roi(current, center_fraction)
    initial_roi = center_roi(initial, center_fraction)

    numerator = np.mean(np.abs(current_roi - initial_roi))
    denominator = np.mean(np.abs(initial_roi)) + eps

    return 100.0 * numerator / denominator


def save_image_cv2(path, image, center_normalizing=1, crop_fraction=0.1):
    image = np.asarray(image, dtype=np.float64)

    if image.ndim != 2:
        raise ValueError('image must be 2D: {}'.format(image.shape))
    if not 0.0 <= crop_fraction < 0.5:
        raise ValueError('crop_fraction must satisfy 0 <= crop_fraction < 0.5')

    h, w = image.shape
    cy = int(np.floor(h * crop_fraction))
    cx = int(np.floor(w * crop_fraction))

    y0 = cy
    y1 = h - cy if cy > 0 else h
    x0 = cx
    x1 = w - cx if cx > 0 else w

    cropped = image[y0:y1, x0:x1].copy()

    if center_normalizing == 1:
        hc, wc = cropped.shape
        yy0 = hc // 4
        yy1 = hc - yy0
        xx0 = wc // 4
        xx1 = wc - xx0
        ref = cropped[yy0:yy1, xx0:xx1]
    else:
        ref = cropped

    ref = ref[np.isfinite(ref)]
    vmin = ref.min()
    vmax = ref.max()

    if vmax > vmin:
        view = (cropped - vmin) / (vmax - vmin)
    else:
        view = np.zeros_like(cropped)

    view[np.isnan(view)] = 0.0
    view[np.isposinf(view)] = 1.0
    view[np.isneginf(view)] = 0.0

    out = np.clip(view * 65535.0, 0, 65535).astype(np.uint16)
    if not cv2.imwrite(path, out):
        raise IOError('Failed to save {}'.format(path))

    return cropped


In [4]:
# ================================================================
# Load PSF and example image
# ================================================================

psf = load_psf(psf_path)
example = load_image(image_path)

print('PSF shape:', psf.shape)
print('PSF sum  :', psf.sum())
print('Image shape:', example.shape)

# Keep the preprocessing expression identical to the existing pipeline.
a = np.ones((noise_level, noise_level), dtype=np.float64)
noise_filter = fftconvolve(example, a, mode='valid') / float(noise_level * noise_level)
background_min = noise_filter.min()

# Intentionally preserves the original expression: image--value == image + value
example_preprocessed = example--background_min
example_normalized = example_preprocessed / example_preprocessed.sum()

print('Background minimum:', background_min)
print('Normalized image sum:', example_normalized.sum())


PSF shape: (41, 41)
PSF sum  : 1.0
Image shape: (612, 564)
Background minimum: 0.04156862745098042
Normalized image sum: 0.9999999999999998


In [ ]:
# ================================================================
# R-L deconvolution
# Save iteration 0 and every subsequent iteration as TIFF.
# Calculate central-region RME relative to iteration 0.
# ================================================================

dcv = example_normalized.copy()
initial_rl = dcv.copy()
psf_flip = psf[::-1, ::-1]

rme_history = [0.0]

# Save iteration 0 (initial value)
save_image_cv2(
    os.path.join(iteration_tif_dir, 'rl_iteration_000.tif'),
    dcv,
    center_normalizing=1,
    crop_fraction=output_crop_fraction
)

print('[SAVED] R-L iteration 0/{}'.format(iteration), flush=True)

for i in range(iteration):
    estimated_blur = correlate_same_fft(dcv, psf)
    relative_blur = example_normalized / (estimated_blur + eps)
    dcv = dcv * correlate_same_fft(relative_blur, psf_flip)

    current_iteration = i + 1

    # RME is calculated from the raw numerical arrays, before display normalization/cropping.
    current_rme = relative_mean_error(
        dcv,
        initial_rl,
        center_fraction=rme_center_fraction,
        eps=eps
    )
    rme_history.append(current_rme)

    # Save every iteration as a TIFF image.
    save_image_cv2(
        os.path.join(
            iteration_tif_dir,
            'rl_iteration_{:03d}.tif'.format(current_iteration)
        ),
        dcv,
        center_normalizing=1,
        crop_fraction=output_crop_fraction
    )

    print(
        '[SAVED] R-L iteration {}/{} | RME = {:.6f}%'.format(
            current_iteration,
            iteration,
            current_rme
        ),
        flush=True
    )

deconvolved_full = dcv.copy()
rme_history = np.asarray(rme_history, dtype=np.float64)

print('Deconvolution complete.')


[SAVED] R-L iteration 0/50
[SAVED] R-L iteration 1/50 | RME = 11.488229%
[SAVED] R-L iteration 2/50 | RME = 17.067304%
[SAVED] R-L iteration 3/50 | RME = 20.259932%
[SAVED] R-L iteration 4/50 | RME = 22.366055%
[SAVED] R-L iteration 5/50 | RME = 23.923519%
[SAVED] R-L iteration 6/50 | RME = 25.169504%
[SAVED] R-L iteration 7/50 | RME = 26.224821%
[SAVED] R-L iteration 8/50 | RME = 27.124045%
[SAVED] R-L iteration 9/50 | RME = 27.894665%
[SAVED] R-L iteration 10/50 | RME = 28.558255%
[SAVED] R-L iteration 11/50 | RME = 29.135872%
[SAVED] R-L iteration 12/50 | RME = 29.646023%
[SAVED] R-L iteration 13/50 | RME = 30.102640%
[SAVED] R-L iteration 14/50 | RME = 30.513152%
[SAVED] R-L iteration 15/50 | RME = 30.884926%
[SAVED] R-L iteration 16/50 | RME = 31.224381%
[SAVED] R-L iteration 17/50 | RME = 31.537161%
[SAVED] R-L iteration 18/50 | RME = 31.827469%
[SAVED] R-L iteration 19/50 | RME = 32.097623%
[SAVED] R-L iteration 20/50 | RME = 32.350326%
[SAVED] R-L iteration 21/50 | RME = 32.587

In [ ]:
# ================================================================
# Save final numerical result and RME data
# ================================================================

final_tif = os.path.join(
    output_dir,
    'example_deconvolution_{}iter_crop10pct.tif'.format(iteration)
)

deconvolved_cropped = save_image_cv2(
    final_tif,
    deconvolved_full,
    center_normalizing=1,
    crop_fraction=output_crop_fraction
)

np.save(
    os.path.join(output_dir, 'example_deconvolution_{}iter_full.npy'.format(iteration)),
    deconvolved_full
)

np.savetxt(
    os.path.join(output_dir, 'example_deconvolution_{}iter_crop10pct.txt'.format(iteration)),
    deconvolved_cropped,
    fmt='%.10e'
)

rme_table = np.column_stack((np.arange(iteration + 1), rme_history))
np.savetxt(
    os.path.join(output_dir, 'RME_vs_RL_iteration.txt'),
    rme_table,
    header='RL_iteration  RME_percent',
    fmt=['%d', '%.10e']
)

np.save(
    os.path.join(output_dir, 'RME_vs_RL_iteration.npy'),
    rme_history
)

print('[SAVED]', final_tif)
print('[SAVED] All iteration TIFFs:', iteration_tif_dir)


In [ ]:
# ================================================================
# RME graph: change from the initial value in central ROI
# ================================================================

rl_steps = np.arange(iteration + 1)

plt.figure(figsize=(8, 5))
plt.plot(rl_steps, rme_history, marker='o', markersize=3)
plt.xlabel('R-L iteration')
plt.ylabel('RME from initial central ROI (%)')
plt.title('R-L evolution relative to initial image')
plt.grid(True, alpha=0.3)
plt.tight_layout()

rme_graph_path = os.path.join(output_dir, 'RME_vs_RL_iteration.png')
plt.savefig(rme_graph_path, dpi=300, bbox_inches='tight')
plt.show()

print('[SAVED]', rme_graph_path)
print('Final RME = {:.6f}%'.format(rme_history[-1]))


In [ ]:
# ================================================================
# Final display
# ================================================================

fig, ax = plt.subplots(1, 3, figsize=(18, 6))
plt.gray()

ax[0].imshow(example_normalized)
ax[0].set_title('R-L initial value')
ax[0].axis('off')

ax[1].imshow(psf)
ax[1].set_title('PSF ({0}x{0})'.format(psf.shape[0]))
ax[1].axis('off')

ax[2].imshow(deconvolved_cropped)
ax[2].set_title('R-L deconvolution ({} iter)'.format(iteration))
ax[2].axis('off')

plt.tight_layout()
plt.show()
